# Price A Password Hash Against An Offline Attacker

Companion notebook for the [Hashing and Password Storage](https://codemia.io/courses/security_engineering/sec_hashing_and_password_storage) lesson on Codemia.

**What you will build**: a small lab for the decisions behind password storage. You will time real key derivation functions
from the standard library on this machine, turn those timings into how much of a leaked user table
an attacker cracks in a day, see exactly what a salt does and does not buy, and plan the migration
from an old hash to a new one, including the accounts that never log in again.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import hashlib
import hmac
import os
import time

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
PASSWORD = b'correct horse battery staple'
print('hashlib algorithms used here: sha256, pbkdf2_hmac, scrypt')
print('all timings below are measured on this machine, one core')

## Concept recap

A password database is designed for the day it leaks. Once an attacker has the table, nothing stops
them from guessing offline, as fast as their hardware allows, so the only defence left is how much
each guess costs. A fast hash such as SHA-256 costs nanoseconds per guess. A password hashing
function, PBKDF2, bcrypt, scrypt or Argon2, is deliberately slow and, for the last two, deliberately
memory hungry, so the attacker's specialised hardware gains less over your server.

Three ideas carry the whole topic. Cost: every guess should take long enough that a day of hardware
buys few guesses. Salt: a random per user value mixed into the hash, so the attacker cannot reuse
work across users. Upgrade: parameters that were right five years ago are too cheap today, so a
system needs a way to move every stored hash to new parameters without knowing the passwords.

## Time real hashes on this machine

In [ ]:
def time_it(fn, repeat=3):
    best = float('inf')
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best

salt = os.urandom(16)
schemes = {}
n = 20_000
schemes['sha256, unsalted'] = time_it(lambda: [hashlib.sha256(PASSWORD).digest() for _ in range(n)]) / n
for iters in (1_000, 10_000, 100_000, 600_000):
    schemes[f'pbkdf2-sha256 x{iters:,}'] = time_it(lambda: hashlib.pbkdf2_hmac('sha256', PASSWORD, salt, iters), repeat=1 if iters > 100_000 else 3)
for log_n in (14, 15, 16):
    schemes[f'scrypt N=2^{log_n}, r=8'] = time_it(lambda: hashlib.scrypt(PASSWORD, salt=salt, n=2 ** log_n, r=8, p=1, maxmem=2 ** 28), repeat=1)

print(f'{"scheme":<28}{"ms per hash":>14}{"hashes/s/core":>16}')
for name, sec in schemes.items():
    print(f'{name:<28}{sec * 1000:>14.4f}{1 / sec:>16,.0f}')

The spread is the point. SHA-256 runs millions of times a second on one core, and PBKDF2 at the
iteration count current guidance recommends for SHA-256, 600,000, runs a few times a second. That
ratio is what you are buying. The same number also shows up as login latency on your own servers, so
the cost you pick is a trade between your CPU bill and the attacker's.

scrypt is different in kind. At N equal to two to the fourteen and r equal to 8 it needs 16 MB per
hash, and the next two rows need 32 and 64 MB. Memory is what makes a GPU or custom chip expensive,
because it has thousands of cores and nowhere near thousands of times the memory bandwidth.

## From hashes per second to accounts cracked per day

In [ ]:
# A leaked table of 100,000 users. Passwords follow a heavy tailed popularity curve over a
# dictionary of two million candidates, which is roughly what breach corpora look like.
USERS = 100_000
DICT = 2_000_000
ranks = np.arange(1, DICT + 1)
weights = 1.0 / ranks ** 0.9
weights /= weights.sum()
user_pw = rng.choice(DICT, size=USERS, p=weights)   # rank of each user's password, 0 is the most popular
never_in_dict = rng.random(USERS) < 0.25            # a quarter use something no dictionary contains

# Assumed attacker advantage over one of your cores, per scheme family. A rig of several GPUs
# is roughly ten thousand times faster at SHA-256 and PBKDF2, and far less at memory hard scrypt.
ADVANTAGE = {'sha256': 10_000, 'pbkdf2': 10_000, 'scrypt': 300}

def cracked_fraction(guesses_per_user):
    hit = (user_pw < guesses_per_user) & ~never_in_dict
    return hit.mean()

DAY = 86_400
print(f'{"scheme":<28}{"attacker guesses/s":>20}{"cracked in a day":>18}')
rows = []
for name, sec in schemes.items():
    family = name.split('-')[0].split(',')[0].split(' ')[0]
    rate = ADVANTAGE[family] / sec
    salted = 'unsalted' not in name
    # salted: the day's guesses are split across every user; unsalted: one pass covers everyone
    per_user = rate * DAY / USERS if salted else rate * DAY
    frac = cracked_fraction(per_user)
    rows.append((name, rate, frac))
    print(f'{name:<28}{rate:>20,.0f}{frac:>17.1%}')

Read the last column as the morning after a breach. The unsalted fast hash loses every account whose
password appears in the dictionary at all, within the day. Even PBKDF2 at a thousand iterations
gives up the most common passwords quickly, because the popular passwords are very popular. The high
cost settings hold most users, but never the people whose password is in the top few hundred,
which is why breached password screening at signup matters as much as the hash.

The advantage numbers are assumptions you should change. Double the attacker and watch which rows
move; the conclusion that memory hard functions shrink the attacker's lead is robust to the exact
values.

In [ ]:
budgets = np.logspace(0, 7, 60)
curve = [cracked_fraction(b) for b in budgets]
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(budgets, curve)
for name, rate, frac in rows:
    if 'unsalted' in name:
        continue
    b = rate * DAY / USERS
    ax.scatter([b], [frac], s=25)
    ax.annotate(name, (b, frac), fontsize=7, xytext=(4, -10), textcoords='offset points')
ax.set_xlabel('guesses per user the attacker can afford')
ax.set_ylabel('fraction of accounts cracked')
ax.set_title('One day of attacker hardware against a salted table')
ax.axhline(0.75, ls='--', lw=0.8, color='gray')
ax.text(1.5, 0.76, 'ceiling: a quarter of users are outside any dictionary', fontsize=7)
plt.tight_layout()
plt.show()

## What a salt buys

In [ ]:
# Store the same 100,000 users two ways and count how many distinct hashes the attacker faces.
sample = user_pw[:20_000]
unsalted = {hashlib.sha256(str(p).encode()).digest() for p in sample}
salts = [os.urandom(16) for _ in sample]
salted = {hashlib.sha256(s + str(p).encode()).digest() for s, p in zip(salts, sample)}
print(f'users: {len(sample):,}')
print(f'distinct unsalted hashes: {len(unsalted):,}  (identical passwords collide, so cracking one cracks them all)')
print(f'distinct salted hashes:   {len(salted):,}  (every user is a separate target)')

top = np.bincount(sample).max()
print(f'the most popular password is shared by {top:,} users in this sample')

A salt does two things and only two. It makes identical passwords look different, so the table no
longer advertises which accounts share the popular password, and it forces the attacker to spend
work per user instead of once for the whole table, which also kills precomputed rainbow tables. It
does nothing to make a single guess more expensive. A salted SHA-256 is still millions of guesses per
second per core against each user. Salt and cost are two separate controls and you need both.

## Upgrading parameters without knowing the passwords

In [ ]:
# Rehash on login: when a user logs in with the old scheme, verify, then store the new hash.
# Users log in at very different rates, and some never come back.
login_rate = rng.choice([1 / 2, 1 / 14, 1 / 60, 1 / 365, 0.0], size=USERS, p=[0.3, 0.3, 0.2, 0.1, 0.1])
days = np.arange(0, 366)
p_logged = 1 - np.exp(-np.outer(days, login_rate))  # probability each user has logged in by day d
migrated = p_logged.mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(days, migrated)
ax.set_xlabel('days since the new parameters shipped')
ax.set_ylabel('accounts on new hash')
ax.set_ylim(0, 1)
ax.set_title('Rehash on login leaves a long tail')
plt.tight_layout()
plt.show()
for d in (7, 30, 90, 365):
    print(f'day {d:>3}: {migrated[d]:.1%} migrated, {1 - migrated[d]:.1%} still on the old hash')

Rehash on login is free and correct, and it never finishes. A year later about a tenth of accounts
have never logged in, and those are exactly the accounts sitting in the table with the weak hash if
it leaks. The fix is to wrap: hash the old hash with the new function now, for every row, without
waiting for anyone. Verification then applies the old function and then the new one. When the user
next logs in, replace the wrapped hash with a clean hash of the password.

In [ ]:
def old_hash(pw, salt):
    return hashlib.pbkdf2_hmac('sha256', pw, salt, 1_000)

def new_hash(data, salt):
    return hashlib.scrypt(data, salt=salt, n=2 ** 14, r=8, p=1)

# migration job: runs once over every stored row, no passwords needed
salt_old, salt_new = os.urandom(16), os.urandom(16)
stored = {'scheme': 'pbkdf2-1k', 'salt': salt_old, 'hash': old_hash(PASSWORD, salt_old)}
stored = {'scheme': 'wrapped', 'salt_old': salt_old, 'salt': salt_new, 'hash': new_hash(stored['hash'], salt_new)}

def verify(row, pw):
    if row['scheme'] == 'wrapped':
        candidate = new_hash(old_hash(pw, row['salt_old']), row['salt'])
    else:
        candidate = new_hash(pw, row['salt'])
    return hmac.compare_digest(candidate, row['hash'])

print('wrapped row verifies the right password:', verify(stored, PASSWORD))
print('wrapped row rejects a wrong password:  ', verify(stored, b'Tr0ub4dor&3'))

# on the next successful login, replace the wrapped row with a clean one
if verify(stored, PASSWORD):
    s = os.urandom(16)
    stored = {'scheme': 'scrypt', 'salt': s, 'hash': new_hash(PASSWORD, s)}
print('after login the row is', stored['scheme'], 'and verifies:', verify(stored, PASSWORD))

Note the comparison. `hmac.compare_digest` takes the same time however many leading bytes match,
which is the defence the CodeExecutor in this lesson builds by hand. A library verify function, such
as the one in argon2-cffi or bcrypt, does the comparison, the salt handling and the parameter parsing
for you, and is what production code should call.

## Exercises

1. Pick a target of 250 ms per login on this machine and find the PBKDF2 iteration count and the
   scrypt N that hit it. How much of the table does each lose in a day under the default advantages?
2. Change the scrypt advantage from 300 to 3,000, which models a better funded attacker. Which row
   moves the most, and does scrypt still beat PBKDF2 at equal login latency?
3. Remove the quarter of users outside any dictionary. What does that do to the ceiling, and what
   does it say about the value of rejecting breached passwords at signup?
4. Extend the migration simulation with a forced password reset for accounts still unmigrated after
   90 days. How many users get the email, and how does that compare with wrapping on day one?

## What to take away

- The table will leak one day. Choose the hash for that day, not for the login path.
- A fast hash is a cracked password for anything in a dictionary. Use a password hashing function
  with a cost tuned to your hardware, and retune it every year or two.
- Salt and cost solve different problems. A salt stops reuse of work across users; only cost makes
  each guess expensive. Memory hardness shrinks the attacker's hardware advantage.
- No hash protects a top hundred password. Screen passwords against breach corpora at signup.
- Rehash on login, and wrap the old hashes immediately so the dormant tail is not left exposed.
- Compare in constant time, and prefer the library's verify function to your own comparison.